# C10-competition-craft — Practice p15

**Type:** scenario · **Difficulty:** core · **Budget:** 90 minutes

**Concepts:** writeup-quality, colab-markdown-solution-authoring, markdown-code-snippets, markdown-math-formulae, colab-coding-submission, cpu-and-gpu-round-boundary

Repair a short Colab response whose prose is in code comments, whose fenced function excerpt is malformed, whose macro-F1 formula is unrendered, whose executable function is singularly named, whose runtime order and downloaded artifact are unverified, and whose Round 1 GPU claim is illegal. Submit a separate artifact for each of the six scored rows.

The executable row must write `p15_contract.csv` with `index=False`, exactly columns `row_id,prediction,is_training_label`, and exactly 12 rows corresponding in order to `X.iloc[40:52]`.

## Row 2 — Colab authoring plan

Provide an ordered `cell_plan` that separates rendered writeup, executable code, fenced communication, rendered mathematics, checks and packaging, and the policy declaration.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
final_model = Pipeline([("scale", StandardScaler()),
                        ("knn", KNeighborsClassifier(n_neighbors=9))])
final_model.fit(X, y)
probe = X.iloc[40:52]
stage_trace = ["setup"]

cell_plan = [
    ("text", "Approach and Intuition writeup"),
    ("code", "imports, data loading, scaled 9-NN fit, and executable predict_labels"),
    ("code", "immutable prediction contract checks"),
    ("text", "rendered fenced predict_labels excerpt"),
    ("text", "rendered macro-F1 derivation and interpretation"),
    ("code", "write and verify p15_contract.csv"),
    ("text", "Round 1 and Round 2 compute-policy declaration"),
]

def predict_labels(X_test):
    predictions = final_model.predict(X_test)
    return pd.Series(predictions, index=X_test.index)

probe_predictions = predict_labels(probe)
stage_trace.append("predict")
contract_checks = {
    "series": isinstance(probe_predictions, pd.Series),
    "length": len(probe_predictions) == len(probe),
    "index": probe_predictions.index.equals(probe.index),
    "vocab": set(probe_predictions.unique()) <= set(np.unique(y)),
}
assert all(contract_checks.values())

In [ ]:
contract_df = pd.DataFrame({
    "row_id": probe.index,
    "prediction": probe_predictions.to_numpy(),
    "is_training_label": np.isin(probe_predictions.to_numpy(), np.unique(y)),
})
contract_df.to_csv("p15_contract.csv", index=False)
stage_trace.append("package")

saved_contract = pd.read_csv("p15_contract.csv")
assert list(saved_contract.columns) == ["row_id", "prediction", "is_training_label"]
assert len(saved_contract) == 12
assert saved_contract["row_id"].tolist() == probe.index.tolist()
assert saved_contract["prediction"].tolist() == probe_predictions.tolist()
assert saved_contract["is_training_label"].tolist() == [True] * 12
assert stage_trace == ["setup", "predict", "package"]

## Row 1 — Approach and Intuition

**Approach.** Fit the named scaled 9-nearest-neighbors recipe on all labeled rows, then expose the exact `predict_labels(X_test)` interface while preserving the input index and training-label vocabulary. No validation score is claimed because this task does not define a validation experiment.

**Intuition.** kNN predicts from distances to nearby training examples. Standardization matters because otherwise a feature with a large numerical scale can dominate those distances even when it is not more informative.

## Row 2 — Authoring diagnosis

Prose written as comments in a code cell is not rendered as a readable submission writeup and is unnecessarily executed by the runtime; it belongs in a text cell with Markdown labels. Conversely, a function displayed inside a fenced block in a text cell is only an excerpt and never defines a callable function, so the contract definition must also appear in a code cell. The malformed double-backtick excerpt neither opens nor closes a proper fenced block, so it will not render as intended. Separating the formula into a text cell and the checks into code cells makes both the explanation and fresh-run evidence auditable.

## Row 3 — Repaired fenced excerpt

```python
def predict_labels(X_test):
    predictions = final_model.predict(X_test)
    return pd.Series(predictions, index=X_test.index)
```

This rendered excerpt communicates the interface; the executable definition remains in the code cell above.

## Row 4 — Rendered macro-F1 derivation

For class $k$, define precision $P_k$, recall $R_k$, and $F_{1,k}=2P_kR_k/(P_k+R_k)$ when the denominator is nonzero. With $K$ classes, macro-F1 is

$$
F_{1,\mathrm{macro}} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}.
$$

The outer average gives every class equal weight, regardless of how many examples that class contributes.

## Row 6 — Round-policy correction

The statement that Round 1 used a GPU because Colab offered one is illegal: available hardware does not override the competition boundary. **Round 1 is CPU only; Round 2 permits Colab L4/GPU.**

### Answer check

The fresh-run trace is exactly setup → predict → package, the prediction Series preserves all 12 row ids and the training vocabulary, and `p15_contract.csv` has the exact required schema, order, and truth flags.